# GPT-2 Training on Google Colab## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted!")

## Install Dependencies

In [ ]:
!pip install torch tiktoken datasets -q
print("Dependencies installed!")

## Import & Training

In [ ]:
import sys
from google.colab import drive
drive.mount('/content/drive')
WORK_DIR = '/content/drive/My Drive/mygpt-2'
sys.path.insert(0, WORK_DIR)

import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, f'{WORK_DIR}/{path}')
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m

config = load_module('config', 'config.py')
train_mod = load_module('train_mod', 'train.py')
gpt_mod = load_module('gpt_mod', 'gpt.py')
main = load_module('main', 'main.py')

GPTConfig = config.GPTConfig
SimpleTokenizer = load_module('tokenizer', 'tokenizer.py').SimpleTokenizer
TextDataset = train_mod.TextDataset
create_dataset = train_mod.create_dataset
GPT = gpt_mod.GPT
train = train_mod.train

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
is_gpu = device.type == 'cuda'
print(f'Using device: {device}')

config = GPTConfig(
    d_model=256 if is_gpu else 128,
    num_heads=8 if is_gpu else 4,
    num_layers=8 if is_gpu else 4,
    max_seq_len=512 if is_gpu else 256,
    batch_size=4 if is_gpu else 2,
    grad_accum_steps=4 if is_gpu else 2,
    max_steps=5000,
    warmup_steps=200,
)

main = load_module('main', 'main.py')
tokenizer = SimpleTokenizer()
texts = main.load_training_data(max_samples=500)
dataset = create_dataset(texts, tokenizer, config.max_seq_len)
print(f'Dataset: {len(dataset):,} batches')

model = GPT(config)
print(f'Model parameters: {model.get_num_params():,}')

model = train(model, dataset, config, device, save_dir='/content/checkpoints')
torch.save(model.state_dict(), '/content/gpt_model.pt')
print('Done!')

## Download Checkpoints

In [ ]:
!zip -r /content/drive/My\ Drive/checkpoints.zip /content/checkpoints/
print("Saved to Drive!")